# 06 — Advanced Vector Database Operations (Qdrant)

**Track:** Intermediate · **Stage:** Retrieval Infrastructure

While we used Chroma previously for simple local RAG, production systems often use specialized databases like **Qdrant**, Pinecone, or Milvus. These systems offer advanced indexing (HNSW), fast payload (metadata) filtering, and native hybrid search.

In this deep dive, you will use **LangChain** and a local **Qdrant** instance (running entirely in-memory for this lab) to enforce strict payload filters and explore collection design.

## Setup: Qdrant and LangChain

We use `QdrantClient` in memory mode so you don't need Docker for this lab.

In [ ]:
# !pip install qdrant-client langchain-qdrant langchain-huggingface

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Filter, FieldCondition, MatchValue

def print_results(results):
    if not results:
        print("[No results found]")
    for i, doc in enumerate(results):
        print(f"[{i+1}] Source: {doc.metadata['source']} | Tenant: {doc.metadata['tenant']}")
        print(f"{doc.page_content}\n")

## 1. Creating the Qdrant Collection

We instantiate Qdrant in-memory and ingest our multi-tenant documents.

In [ ]:
client = QdrantClient(":memory:")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

corpus = [
    Document(
        page_content="Acme Runbook: If checkout fails, verify the Stripe gateway connection.",
        metadata={"source": "acme_checkout.md", "tenant": "acme"}
    ),
    Document(
        page_content="Globex Architecture: The checkout service is written in Go and uses a Redis cache.",
        metadata={"source": "globex_checkout.md", "tenant": "globex"}
    )
]

# LangChain integration for Qdrant
qdrant = QdrantVectorStore(
    client=client, 
    collection_name="support_docs", 
    embedding=embeddings
)

# Add documents
qdrant.add_documents(corpus)
print("Documents indexed into Qdrant in-memory collection.")

## 2. Advanced Payload Filtering (Qdrant Native)

Unlike Chroma's simple dictionary filters, Qdrant offers a rich strongly-typed filtering syntax. We will construct a `Filter` object using Qdrant's native models and pass it through LangChain.

In [ ]:
question = "Tell me about the checkout service."

print("--- Naive Search (Leaks Data) ---")
print_results(qdrant.similarity_search(question, k=2))

print("--- Secure Search (Acme Tenant Only) ---")
# Construct a native Qdrant filter
acme_filter = Filter(
    must=[
        FieldCondition(
            key="metadata.tenant", 
            match=MatchValue(value="acme")
        )
    ]
)

# Pass the Qdrant filter to LangChain
secure_results = qdrant.similarity_search(question, k=2, filter=acme_filter)
print_results(secure_results)

## 3. Operations: Updating and Deleting Vectors

In production, documents become stale. You must be able to delete vectors without rebuilding the whole collection.

In [ ]:
new_doc = Document(
        page_content="Acme Runbook (V2): The Stripe gateway was replaced with Adyen. If checkout fails, check Adyen.",
        metadata={"source": "acme_checkout.md", "tenant": "acme", "version": 2}
)

# In a real scenario, you would track the Qdrant point IDs associated with 'acme_checkout.md' 
# and explicitly delete them using client.delete(). LangChain abstractions often hide this, 
# but enterprise RAG requires strict ID management.
qdrant.add_documents([new_doc])

# Note: Without deleting the old vector, we now have contradictory information in the DB.
print("--- Search after bad update (Contradictory Results) ---")
print_results(qdrant.similarity_search("What gateway does checkout use?", k=2, filter=acme_filter))

## Reflection

1. **Payload Schema:** Vector DBs are still databases. If your payload (metadata) lacks a `source_id` or `version`, you cannot easily prune stale data.
2. **Vector vs Document:** A document (e.g. `policy.pdf`) might be split into 50 chunks (50 vectors). If `policy.pdf` is deleted by a user, your system must execute a metadata filter deletion query (e.g. `delete where metadata.source == 'policy.pdf'`) to purge all 50 vectors instantly.